In [ ]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [ ]:
!pip install -q tensorflow-recommenders

In [ ]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

In [ ]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [ ]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [ ]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [ ]:
tf_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            key:
            training_df[
                key
            ].values

            for key in
            training_df.columns
        }
    )
)

BATCH_SIZE = 2048

train = (
    tf_dataset
    .take(
        int(
            0.8 *
            len(training_df)
        )
    )
    .shuffle(100000)
    .batch(BATCH_SIZE)
    .cache()
)

test = (
    tf_dataset
    .skip(
        int(
            0.8 *
            len(training_df)
        )
    )
    .batch(BATCH_SIZE)
    .cache()
)

In [ ]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [ ]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [ ]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

In [ ]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [ ]:
model = (
    TwoTowerModel()
)

In [ ]:
model.compile(

    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [ ]:
for batch in train.take(1):

    model.compute_loss(
        batch,
        training=False
    )

print("Model built!")

In [ ]:
#
history = model.fit(

    train,

    validation_data=test,

    epochs=3
)

In [ ]:
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-tower-2/two_tower_weights_2.weights.h5"
)

print(
    "Epoch 2 weights loaded!"
)

In [ ]:
history = model.fit(

    train,

    validation_data=test,

    initial_epoch=2,
    epochs=3
)

In [ ]:
#
model.save_weights(
    "/kaggle/working/two_tower_weights.weights.h5"
)

print(
    "Model saved!"
)